# Pipeline Landing → Bronze

Ingestão de dados cinematográficos (CSVs) e cotação do dólar (API BCB) para a camada Bronze.

In [0]:
spark.sql("CREATE catalog IF NOT EXISTS cinedata_lakehouse")
spark.sql("CREATE SCHEMA IF NOT EXISTS cinedata_lakehouse.bronze")
spark.sql("USE CATALOG cinedata_lakehouse")
spark.sql("USE SCHEMA bronze")

# Ingestão de Dados Cinematográficos (CSV)

Leitura dos arquivos CSV do volume e carga append nas tabelas Bronze com timestamp de ingestão.

In [0]:
from pyspark.sql.functions import current_timestamp



arquivos_tabelas = {
    "movies_info_TMDB_IMDB.csv": "tb_movies_info",
    "movies_financials_IMDB_TMDB.csv": "tb_movies_financials",
    "movies_metrics_IMDB_TMDB.csv": "tb_movies_metrics",
    "credits_and_tags_IMDB_TMDB.csv": "tb_credits_and_tags",
    "movies_reviews.csv": "tb_movies_reviews"
}

caminho_volume = "/Volumes/cinedata_lakehouse/bronze/input/"

for arquivo, tabela in arquivos_tabelas.items():
    df_raw = (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(f"{caminho_volume}{arquivo}")
    )
    df_bronze = df_raw.withColumn("ingestion_datetime", current_timestamp())
    (
        df_bronze.write
        .format("delta")
        .mode("append")
        .saveAsTable(f"bronze.{tabela}")
    )

# Ingestão de Cotação do Dólar (API BCB)

Criação dos widgets de data e consulta à API PTAX do Banco Central para ingestão das cotações.

In [0]:
from datetime import datetime,timedelta
hoje = datetime.now()
sete_dias_atras = hoje - timedelta(days=7)
data_inicio_formatada = sete_dias_atras.strftime("%m-%d-%Y")
data_fim_formatada = hoje.strftime("%m-%d-%Y")
dbutils.widgets.text("data_inicio",data_inicio_formatada,"Data Inicio")
dbutils.widgets.text("data_fim",data_fim_formatada,"Data Fim")

In [0]:
import requests
from pyspark.sql.functions import current_timestamp
data_fim_formatada = dbutils.widgets.get("data_fim")
data_inicio_formatada = dbutils.widgets.get("data_inicio") 
url = f"https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)?@dataInicial='{data_inicio_formatada}'&@dataFinalCotacao='{data_fim_formatada}'&$select=dataHoraCotacao,cotacaoCompra&$format=json"

responde = requests.get(url)
cotacao = responde.json()
cotacao = cotacao['value']

if cotacao:
    df_cotacao_raw = spark.createDataFrame(cotacao)
    df_cotacao_bronze = df_cotacao_raw.withColumn("ingestion_datetime", current_timestamp())

    (
        df_cotacao_bronze.write
        .format("delta")
        .mode("append")
        .saveAsTable("bronze.tb_cotacao_dolar")
    )
    print(f"Carregou {len(cotacao)} cotações em bronze.tb_cotacao_dolar")
else:
    print(f"Nenhuma data disponivel entre {data_inicio_formatada} e {data_fim_formatada}")

